# Demo 4: Building Agent Tools on Databricks

**Module 5A - Hour 4: Agent Tools (Topics 4.1-4.6)**

This demo focuses on **creating tools** for AI agents — NOT on building agents themselves (that's Demo 5).

We cover:
* Creating SQL UC functions as tools
* Creating Python UC functions as tools
* Tool discovery via Unity Catalog
* How agents find and use tools (conceptually)
* AI Playground for tool testing
* AI Gateway for production guardrails and routing

Each section has **Notes** (concept + rationale) followed by a **Demo** cell with live code.

In [0]:
%sql
-- SETUP: Create catalog, schema, sample data, and UC functions
-- We create a dedicated catalog/schema for this demo with a
-- customer_orders table that simulates an e-commerce system.
-- UC functions registered here serve as governed agent tools.

CREATE CATALOG IF NOT EXISTS module5a_demo4;
CREATE SCHEMA IF NOT EXISTS module5a_demo4.agent_tools;

-- Sample data table for agent tool demos
CREATE OR REPLACE TABLE module5a_demo4.agent_tools.customer_orders (
  order_id     STRING NOT NULL,
  customer_id STRING NOT NULL,
  customer_tier STRING,
  order_total  DOUBLE,
  status       STRING,
  order_date   DATE,
  CONSTRAINT pk_customer_orders PRIMARY KEY (order_id)
) TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

INSERT INTO module5a_demo4.agent_tools.customer_orders VALUES
('ORD-001', 'CUST-001', 'premium', 249.99, 'shipped',    '2026-09-15'),
('ORD-002', 'CUST-001', 'premium', 89.50,  'delivered',  '2026-09-18'),
('ORD-003', 'CUST-002', 'standard', 159.00, 'pending',   '2026-09-20'),
('ORD-004', 'CUST-002', 'standard', 45.00,  'cancelled', '2026-09-22'),
('ORD-005', 'CUST-003', 'basic',   310.75, 'shipped',    '2026-09-24'),
('ORD-006', 'CUST-003', 'basic',   78.25,  'processing', '2026-09-25'),
('ORD-007', 'CUST-001', 'premium', 199.50, 'delivered',  '2026-09-10'),
('ORD-008', 'CUST-002', 'standard', 525.00, 'pending',   '2026-09-25');

SELECT * FROM module5a_demo4.agent_tools.customer_orders ORDER BY order_id;

order_id,customer_id,customer_tier,order_total,status,order_date
ORD-001,CUST-001,premium,249.99,shipped,2026-09-15
ORD-002,CUST-001,premium,89.5,delivered,2026-09-18
ORD-003,CUST-002,standard,159.0,pending,2026-09-20
ORD-004,CUST-002,standard,45.0,cancelled,2026-09-22
ORD-005,CUST-003,basic,310.75,shipped,2026-09-24
ORD-006,CUST-003,basic,78.25,processing,2026-09-25
ORD-007,CUST-001,premium,199.5,delivered,2026-09-10
ORD-008,CUST-002,standard,525.0,pending,2026-09-25


In [0]:
%sql
-- SETUP: Register UC functions as governed agent tools
-- These functions are discovered and called by AI agents.
-- Unity Catalog governs access: GRANT/REVOKE EXECUTE on each function.

-- Tool 1: Get order status by order ID
CREATE OR REPLACE FUNCTION module5a_demo4.agent_tools.get_order_status(order_id STRING)
RETURNS STRING
COMMENT 'Returns the current status of an order (shipped, delivered, pending, cancelled, processing)'
RETURN (SELECT max(status) FROM module5a_demo4.agent_tools.customer_orders
       WHERE order_id = get_order_status.order_id);

-- Tool 2: Calculate discount based on order total and customer tier
CREATE OR REPLACE FUNCTION module5a_demo4.agent_tools.calculate_discount(order_total DOUBLE, customer_tier STRING)
RETURNS DOUBLE
COMMENT 'Calculates discount amount: premium=15%, standard=5%, basic=0%'
RETURN CASE
  WHEN customer_tier = 'premium'  THEN round(order_total * 0.15, 2)
  WHEN customer_tier = 'standard' THEN round(order_total * 0.05, 2)
  ELSE 0.0
END;

-- Tool 3: Get order count for a customer
CREATE OR REPLACE FUNCTION module5a_demo4.agent_tools.get_order_count(customer_id STRING)
RETURNS INT
COMMENT 'Returns the total number of orders for a customer'
RETURN SELECT count(*) FROM module5a_demo4.agent_tools.customer_orders
       WHERE customer_id = get_order_count.customer_id;

-- Tool 4: Get total spending for a customer
CREATE OR REPLACE FUNCTION module5a_demo4.agent_tools.get_total_spending(customer_id STRING)
RETURNS DOUBLE
COMMENT 'Returns total spending for a customer across all orders'
RETURN SELECT coalesce(sum(order_total), 0.0) FROM module5a_demo4.agent_tools.customer_orders
       WHERE customer_id = get_total_spending.customer_id;

-- Verify all tools work
SELECT
  module5a_demo4.agent_tools.get_order_status('ORD-001') AS ord_001_status,
  module5a_demo4.agent_tools.calculate_discount(100.0, 'premium') AS premium_discount,
  module5a_demo4.agent_tools.get_order_count('CUST-001') AS cust_001_orders,
  module5a_demo4.agent_tools.get_total_spending('CUST-003') AS cust_003_spending;

ord_001_status,premium_discount,cust_001_orders,cust_003_spending
shipped,15.0,3,389.0


## 4.1 : What are Agent Tools?

### Concepts
* **Agent tools** are functions that AI agents can call to interact with data, systems, and external APIs.
* **Why tools?** LLMs alone can only generate text. Tools let agents take actions: query data, call APIs, execute code.
* **Unity Catalog Functions** are the Databricks way to create governed, discoverable agent tools.
* **Two types**:
  * **SQL functions**: Query tables, aggregate data, filter records
  * **Python functions**: Call external APIs, transform data, complex logic

**Key principle**: Tools extend what agents can do beyond text generation. Without tools, an agent is just a chatbot. With tools, it becomes an action-taking system.

In [0]:
# 4.1 Demo: Test the UC functions we created
# These are the tools agents will call.

print("=== Testing UC Functions as Agent Tools ===")
print()

# Tool 1: Get order status
status = spark.sql("""
  SELECT module5a_demo4.agent_tools.get_order_status('ORD-001')
""").collect()[0][0]
print(f"1. get_order_status('ORD-001') -> '{status}'")

# Tool 2: Calculate discount
discount = spark.sql("""
  SELECT module5a_demo4.agent_tools.calculate_discount(200.0, 'premium')
""").collect()[0][0]
print(f"2. calculate_discount(200.0, 'premium') -> ${discount:.2f}")

# Tool 3: Get order count
count = spark.sql("""
  SELECT module5a_demo4.agent_tools.get_order_count('CUST-001')
""").collect()[0][0]
print(f"3. get_order_count('CUST-001') -> {count} orders")

# Tool 4: Get total spending
spending = spark.sql("""
  SELECT module5a_demo4.agent_tools.get_total_spending('CUST-003')
""").collect()[0][0]
print(f"4. get_total_spending('CUST-003') -> ${spending:,.2f}")

print()
print("All tools work. Agents can now call these functions.")

=== Testing UC Functions as Agent Tools ===

1. get_order_status('ORD-001') -> 'shipped'
2. calculate_discount(200.0, 'premium') -> $30.00
3. get_order_count('CUST-001') -> 3 orders
4. get_total_spending('CUST-003') -> $389.00

All tools work. Agents can now call these functions.


## 4.2 : Creating Python UC Functions

### Concepts
* **When to use Python functions**: External API calls, complex transformations, integrations
* **When to use SQL functions**: Data queries, aggregations, filtering
* **Python function requirements**:
  * Type hints for parameters and return value
  * Docstring becomes the tool description (critical for agent discovery)
  * Register with `CREATE FUNCTION` using Python code

### Example Python function
```python
def validate_order(order_id: str) -> bool:
    """Checks if an order ID is valid and exists in the system."""
    # Logic here
    return True
```

> The docstring is what the agent sees. Make it clear and action-oriented.

In [0]:
%sql
-- 4.2b Demo: Python UC function with real business logic
-- Python functions can do things SQL can't: conditional logic, string parsing, math

-- Python function: Assess risk based on order value and customer tier
CREATE OR REPLACE FUNCTION module5a_demo4.agent_tools.assess_order_risk(order_total DOUBLE, customer_tier STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Assesses the fraud risk level of an order (low, medium, high) based on amount and customer tier'
AS $$
  if customer_tier == 'premium':
    threshold = 1000.0
  elif customer_tier == 'standard':
    threshold = 500.0
  else:
    threshold = 200.0
  
  if order_total > threshold:
    return f"high risk: order ${order_total:.2f} exceeds ${threshold:.2f} threshold for {customer_tier} tier"
  elif order_total > threshold * 0.7:
    return f"medium risk: order ${order_total:.2f} near ${threshold:.2f} threshold for {customer_tier} tier"
  else:
    return f"low risk: order ${order_total:.2f} within normal range for {customer_tier} tier"
$$;

-- Python function: Generate a customer summary from raw data
CREATE OR REPLACE FUNCTION module5a_demo4.agent_tools.generate_customer_summary(customer_id STRING, total_orders INT, total_spending DOUBLE)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Generates a human-readable customer summary with tier classification and average order value'
AS $$
  avg_order = total_spending / total_orders if total_orders > 0 else 0.0
  if total_spending > 1000:
    classification = "VIP"
  elif total_spending > 500:
    classification = "valued"
  else:
    classification = "standard"
  return f"Customer {customer_id}: {total_orders} orders, ${total_spending:.2f} total, ${avg_order:.2f} avg/order, classified as {classification}"
$$;

-- Test both Python functions
SELECT
  module5a_demo4.agent_tools.assess_order_risk(525.00, 'standard') AS risk_test_1,
  module5a_demo4.agent_tools.assess_order_risk(310.75, 'basic') AS risk_test_2,
  module5a_demo4.agent_tools.generate_customer_summary('CUST-001', 3, 538.99) AS summary_test;

risk_test_1,risk_test_2,summary_test
high risk: order $525.00 exceeds $500.00 threshold for standard tier,high risk: order $310.75 exceeds $200.00 threshold for basic tier,"Customer CUST-001: 3 orders, $538.99 total, $179.66 avg/order, classified as valued"


## 4.2c : @function_tool Decorator (Code-First Approach)

### Concepts
The `@function_tool` decorator is from the **OpenAI Agents SDK** and is supported on Databricks for creating agent tools:

* **Import**: `from agents import function_tool`
* **Install**: `pip install openai-agents`
* **What it does**: Takes a Python function with type hints and a docstring, and generates a tool schema the LLM understands.
* **Type hints** become parameter types in the tool schema (JSON Schema format).
* **Docstring** becomes the tool description the LLM reads to decide when to call it.
* **Return type** becomes the tool's output type.

### UC Functions vs @function_tool

| Aspect | UC Functions | @function_tool |
|---|---|---|
| **Governance** | Unity Catalog (GRANT/REVOKE) | Code-level (no governance) |
| **Discovery** | `information_schema.routines` | Manual registration |
| **Best for** | Production, governed tools | Prototyping, local functions |
| **Language** | SQL or Python (in UC) | Python only |
| **Install** | Built into Databricks | `pip install openai-agents` |

> Use `@function_tool` for rapid prototyping in a notebook. Migrate to UC functions for production governance.

In [0]:
# 4.2c Demo: @function_tool from the OpenAI Agents SDK
# The real decorator converts type hints + docstring into a tool schema.
# Install: pip install openai-agents

%pip install openai-agents --quiet 2>&1 | tail -1

from agents import function_tool

# Define tools using the real @function_tool decorator
# Type hints become parameter types, docstring becomes tool description
@function_tool
def assess_risk(order_total: float, customer_tier: str) -> str:
    """Assesses the fraud risk level of an order based on amount and customer tier"""
    return spark.sql(f"""
      SELECT module5a_demo4.agent_tools.assess_order_risk({order_total}, '{customer_tier}')
    """).collect()[0][0]

@function_tool
def get_order_status(order_id: str) -> str:
    """Returns the current status of an order (shipped, delivered, pending, cancelled, processing)"""
    return spark.sql(f"""
      SELECT module5a_demo4.agent_tools.get_order_status('{order_id}')
    """).collect()[0][0]

@function_tool
def get_total_spending(customer_id: str) -> float:
    """Returns total spending for a customer across all orders"""
    return spark.sql(f"""
      SELECT module5a_demo4.agent_tools.get_total_spending('{customer_id}')
    """).collect()[0][0]

# Show the auto-generated tool schemas
print("=== @function_tool: Auto-Generated Tool Schemas ===")
print()
for tool in [assess_risk, get_order_status, get_total_spending]:
    print(f"Tool: {tool.name}")
    print(f"  Description: {tool.description}")
    print(f"  Params JSON Schema: {tool.params_json_schema}")
    print()

# Execute the underlying UC functions directly
print("=== Tool Execution (via UC functions) ===")
r1 = spark.sql("SELECT module5a_demo4.agent_tools.assess_order_risk(525.0, 'standard')").collect()[0][0]
print(f"  assess_risk(525.0, 'standard') -> {r1}")
r2 = spark.sql("SELECT module5a_demo4.agent_tools.get_order_status('ORD-001')").collect()[0][0]
print(f"  get_order_status('ORD-001') -> {r2}")
r3 = spark.sql("SELECT module5a_demo4.agent_tools.get_total_spending('CUST-003')").collect()[0][0]
print(f"  get_total_spending('CUST-003') -> ${r3:,.2f}")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
=== @function_tool: Auto-Generated Tool Schemas ===

Tool: assess_risk
  Description: Assesses the fraud risk level of an order based on amount and customer tier
  Params JSON Schema: {'properties': {'order_total': {'title': 'Order Total', 'type': 'number'}, 'customer_tier': {'title': 'Customer Tier', 'type': 'string'}}, 'required': ['order_total', 'customer_tier'], 'title': 'assess_risk_args', 'type': 'object', 'additionalProperties': False}

Tool: get_order_status
  Description: Returns the current status of an order (shipped, delivered, pending, cancelled, processing)
  Params JSON Schema: {'properties': {'order_id': {'title': 'Order Id', 'type': 'string'}}, 'required': ['order_id'], 'title': 'get_order_status_args', 'type': 'object', 'additionalProperties': False}

Tool: get_total_spending
  Description: Returns total spending for a customer across all orders
 

## 4.3 : Tool Discovery

### Concepts
* **How agents find tools**: Agents query Unity Catalog's `information_schema.routines` to discover available functions.
* **Tool metadata**:
  * `routine_name`: The function name (becomes the tool name)
  * `comment`: The tool description the LLM reads to decide when to call it
  * `routine_type`: FUNCTION or PROCEDURE
  * `data_type`: Return type
* **Why COMMENT matters**: The LLM uses the comment to understand what the tool does. Write clear, action-oriented descriptions.
* **Governance**: `GRANT EXECUTE` controls which users can call which tools. Agents respect UC permissions.

> An agent's tool selection quality depends entirely on your function COMMENT. Write them like instructions for an LLM: "Returns the X", "Calculates Y", "Checks whether Z".

In [0]:
%sql
-- 4.3 Demo: List and inspect registered UC functions as agent tools
-- Agents discover tools by querying the UC information schema.
-- Each function's comment serves as the tool description for the LLM.

-- List all functions in our schema (this is what an agent does to discover tools)
SELECT
  routine_name AS function_name,
  comment AS tool_description,
  routine_type
FROM module5a_demo4.information_schema.routines
WHERE routine_schema = 'agent_tools'
ORDER BY function_name;

-- Show the full function definition for one tool (agents use this to understand parameters)
DESCRIBE FUNCTION EXTENDED module5a_demo4.agent_tools.calculate_discount;

function_desc
Function: module5a_demo4.agent_tools.calculate_discount
Type: SCALAR
Input: order_total DOUBLE
customer_tier STRING COLLATE UTF8_BINARY
Returns: DOUBLE
"Comment: Calculates discount amount: premium=15%, standard=5%, basic=0%"
Collation: UTF8_BINARY
Deterministic: true
Data Access: CONTAINS SQL
Configs: runtime.databricks.sql.functions.counter_diff.enabled=true


## 4.4 : How Agents Use Tools

### Concepts
The agent workflow:
1. **User asks a question**: "What is the status of order ORD-001?"
2. **Agent discovers tools**: Queries UC `information_schema.routines` to list available functions
3. **Agent reads tool descriptions**: The `comment` field tells the LLM what each tool does
4. **Agent decides which tool to call**: LLM picks the best match based on the question and tool descriptions
5. **Agent executes the tool**: Calls the UC function with parameters
6. **Agent observes the result**: Reads the tool output
7. **Agent responds**: Synthesizes a natural language answer using the tool result

**Key insight**: Tools are useless if the agent doesn't know when to call them. Clear, action-oriented COMMENT fields are critical.

> The demo below shows this workflow in action.

In [0]:
# 4.4 Demo: How agents use tools
# Agents read tool descriptions, decide which to call, execute, and respond.

print("=== How Agents Use Tools ===")
print()

# The agent workflow:
# 1. User asks a question
# 2. Agent reads tool descriptions from UC
# 3. Agent decides which tool to call based on the COMMENT
# 4. Agent executes the tool
# 5. Agent responds using the tool result

print("Step 1: User asks: 'What is the status of order ORD-001?'")
print()

print("Step 2: Agent queries UC to find available tools")
tools = spark.sql("""
  SELECT routine_name, comment
  FROM module5a_demo4.information_schema.routines
  WHERE routine_schema = 'agent_tools'
  ORDER BY routine_name
""").collect()
for row in tools:
    print(f"  - {row['routine_name']}: {row['comment'][:60]}...")
print()

print("Step 3: Agent reads 'get_order_status' COMMENT and decides it's the right tool")
print()

print("Step 4: Agent calls get_order_status('ORD-001')")
status = spark.sql("SELECT module5a_demo4.agent_tools.get_order_status('ORD-001')").collect()[0][0]
print(f"  Tool returned: '{status}'")
print()

print("Step 5: Agent responds to user")
response = spark.sql(f"""
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'The tool returned: {status}. Respond to the user naturally: What is the status of order ORD-001?',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 50)
  ) AS response
""").collect()[0][0]
print(f"  Agent: {response.strip()}")

=== How Agents Use Tools ===

Step 1: User asks: 'What is the status of order ORD-001?'

Step 2: Agent queries UC to find available tools
  - assess_order_risk: Assesses the fraud risk level of an order (low, medium, high...
  - calculate_discount: Calculates discount amount: premium=15%, standard=5%, basic=...
  - format_order_summary: Formats an order summary as a human-readable string...
  - generate_customer_summary: Generates a human-readable customer summary with tier classi...
  - get_order_count: Returns the total number of orders for a customer...
  - get_order_status: Returns the current status of an order (shipped, delivered, ...
  - get_total_spending: Returns total spending for a customer across all orders...

Step 3: Agent reads 'get_order_status' COMMENT and decides it's the right tool

Step 4: Agent calls get_order_status('ORD-001')
  Tool returned: 'shipped'

Step 5: Agent responds to user
  Agent: The status of your order ORD-001 is "shipped". This means that it has b

## 4.5 : AI Playground

### Concepts
* The **AI Playground** is a no-code interface for testing prompts, models, and tools before writing code.
* **What you can test**:
  * Compare different foundation models side-by-side
  * Adjust temperature, max_tokens, and system prompts
  * Test UC functions as tools (the Playground discovers them automatically)
  * Save prompts as templates for reuse
* **Workflow**: Prototype in Playground -> verify behavior -> wire into production code or agent.
* **Key benefit**: Zero setup - the Playground has access to all foundation models and UC functions you can access.

> The Playground is the fastest way to validate that a prompt + tool combination works before committing to agent code.

## 4.6 : Unity AI Gateway

### Concepts
The **Unity AI Gateway** is a managed service that sits between your applications and LLM endpoints, providing:

* **Guardrails**: Filter harmful content in both requests (input) and responses (output). Detect PII, block prompt injections.
* **Rate limits**: Control requests per minute, per user, or per token. Prevents abuse and controls costs.
* **Usage tracking**: Every request is logged to an inference table in Unity Catalog. Query usage by user, model, token count, or cost.
* **Traffic splitting & fallback routing**: Route traffic across multiple models. If the primary model fails, automatically fall back to a secondary.
  * Example: 80% to `databricks-meta-llama-3-3-70b-instruct`, 20% to `databricks-llama-4-maverick`
  * Fallback: If primary returns 5xx, route to `databricks-gpt-oss-20b`
* **Governed**: All gateway configurations are UC entities with access controls.

> The AI Gateway is the production layer between your agent and the model. Without it, you have no guardrails, no rate limits, and no usage visibility.

In [0]:
# 4.6 Demo: AI Gateway concepts - guardrails, routing, usage tracking
# The AI Gateway sits between applications and LLM endpoints.
# Here we demonstrate the concepts that the Gateway provides.

print("=== Unity AI Gateway: Key Features ===")
print()

# 1. Guardrails - content filtering (the Gateway filters requests/responses)
print("1. Guardrails: Filter harmful content in requests and responses")
print("   - Input guardrails: detect PII, block prompt injection")
print("   - Output guardrails: filter harmful responses")
print("   - Configured per gateway endpoint")
print()

# 2. Usage tracking - monitor token consumption
print("2. Usage Tracking: Every request logged to UC inference table")
print("   - Track: user, model, token count, latency, cost")
print("   - Query: SELECT user, sum(tokens) FROM inference_table GROUP BY user")
print()

# 3. Rate limiting - control request volume
print("3. Rate Limits: Control requests per minute/user")
print("   - Example: 100 requests/min for standard users, 1000/min for premium")
print("   - Prevents abuse and controls costs")
print()

# 4. Traffic splitting & fallback routing
print("4. Traffic Splitting & Fallback Routing:")
print("   Primary:   databricks-meta-llama-3-3-70b-instruct (80%)")
print("   Secondary: databricks-llama-4-maverick (20%)")
print("   Fallback:  databricks-gpt-oss-20b (if primary fails)")
print()

# Show a real LLM call that would go through the Gateway in production
print("=== Example: LLM call (would route through Gateway in production) ===")
response = spark.sql("""
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'In one sentence, what is the Unity AI Gateway?',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 80)
  ) AS response
""").collect()[0][0]
print(f"  {response}")

=== Unity AI Gateway: Key Features ===

1. Guardrails: Filter harmful content in requests and responses
   - Input guardrails: detect PII, block prompt injection
   - Output guardrails: filter harmful responses
   - Configured per gateway endpoint

2. Usage Tracking: Every request logged to UC inference table
   - Track: user, model, token count, latency, cost
   - Query: SELECT user, sum(tokens) FROM inference_table GROUP BY user

3. Rate Limits: Control requests per minute/user
   - Example: 100 requests/min for standard users, 1000/min for premium
   - Prevents abuse and controls costs

4. Traffic Splitting & Fallback Routing:
   Primary:   databricks-meta-llama-3-3-70b-instruct (80%)
   Secondary: databricks-llama-4-maverick (20%)
   Fallback:  databricks-gpt-oss-20b (if primary fails)

=== Example: LLM call (would route through Gateway in production) ===
  The Unity AI Gateway is a cloud-based platform that enables developers to integrate artificial intelligence (AI) and machine l

## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| 4.1 | What are agent tools? | Tools let agents take actions beyond text generation; UC functions = governed tools |
| 4.2 | Creating Python UC functions | Type hints + docstring = tool schema; use Python for APIs/transforms, SQL for data |
| 4.3 | Tool discovery via UC | Agents query `information_schema.routines`; COMMENT field is the tool description |
| 4.4 | How agents use tools | Agent workflow: discover → read descriptions → decide → execute → respond |
| 4.5 | AI Playground | No-code prototyping of prompts, models, and UC tools before writing agent code |
| 4.6 | Unity AI Gateway | Guardrails, rate limits, usage tracking, traffic splitting & fallback routing |

### Key principles
* **Tools = agent capabilities**: Without tools, an agent is just a chatbot. With tools, it can act.
* **UC functions are governed**: `GRANT EXECUTE` controls access; audit logs track all tool calls.
* **COMMENT quality = tool selection quality**: Clear descriptions help agents pick the right tool.
* **Prototype in Playground, govern with UC, scale with Gateway**: The Databricks agent toolkit.

In [0]:
# CLEANUP: Drop all resources created in this demo
print("Dropping UC functions...")
for fn in ['get_order_status', 'calculate_discount', 'get_order_count', 'get_total_spending', 'format_order_summary']:
    spark.sql(f"DROP FUNCTION IF EXISTS module5a_demo4.agent_tools.{fn}")
print("  Functions dropped")

print("Dropping schema and catalog...")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo4.agent_tools CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo4 CASCADE")
print("  Schema and catalog dropped")

print("\nCleanup complete!")